# CIFAR-10 Deep Learning Assignment

CIFAR-10 has:

50,000 training images each belonging to 1 of the following 10 classes:

*   plane
*   cat
*   deer
*   dog
*   frog
*   horse
*   ship
*   truck

Image size: 32×32

In [ ]:
!pip install torch torchvision matplotlib -q

## 1. Imports

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader, Subset
import matplotlib.pyplot as plt
import numpy as np
from sklearn.model_selection import KFold
import seaborn as sns
from sklearn.metrics import confusion_matrix

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

# 10 classes in CIFAR-10 dataset
classes = ('plane',
           'car',
           'bird',
           'cat',
           'deer',
           'dog',
           'frog',
           'horse',
           'ship',
           'truck')

## 2. Data Loading & Augmentation:



*   **Augmentation**: As CIFAR-10 dataset is a small dataset: if we train without augmentation, model might memorize exact pixel layouts and overfit

    Hence, we do:
    *   **RandomHorizontalFlip:** Without flipping, the model may associate say a car to a specific left-facing pixel pattern
          But flipping teaches:
          car facing left might also be car and
          car facing right might also be car

          So it learns object identity, not orientation.
    *  **RandomCrop**: Neural networks are not naturally translation invariant.

        If a dog appears:

        Slightly left in training and slightly right in testing

        Then without crop augmentation:
        Model may treat them differently.
    *   **ColorJitter**: Randomly changes: brightness, color and saturation to make model robust to lighting conditions and will also prevent overfitting to specific color intensity

In [ ]:
CIFAR_MEAN = (0.4914, 0.4822, 0.4465)
CIFAR_STD  = (0.2023, 0.1994, 0.2010)

# transforms.Compose applies transformations one after another in sequence:

transform_train = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomCrop(32, padding=4),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(), # Converting Image to Tensor: Pytorch requires tensor in form of [Channels, Height, Width]
    transforms.Normalize(CIFAR_MEAN, CIFAR_STD)
])

transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(CIFAR_MEAN, CIFAR_STD)
])

full_trainset = torchvision.datasets.CIFAR10(root='./data', train=True,  download=True, transform=transform_train)
testset       = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=transform_test)

# Loaders get the dataset , loads it into mini batches, shuffles it (in case of traindata only here : shuffle=True)
trainloader = DataLoader(full_trainset, batch_size=128, shuffle=True,  num_workers=2, pin_memory=True)
testloader  = DataLoader(testset,       batch_size=128, shuffle=False, num_workers=2, pin_memory=True)

print(f'Train samples: {len(full_trainset)} | Test samples: {len(testset)}')

## 3. Utility Functions
###  3a. Training Loop (returns history for plotting):

The loop returns history: [train loss, validation loss, validation accuracy]

Some of the optimisations that helped are:

*   **Label Smoothing** added before calculating Loss:  it is a regularization technique used in machine learning to reduce overfitting,
*   **AdamW Optimiser** used:

      gradient_with_decay = gradient + λ * weight
      scaled_update = adaptive_scale(gradient_with_decay)

      Because weight decay gets added to the gradient before the adaptive scaling, it ends up being scaled differently for each parameter depending on how large or noisy that parameter's gradients have been historically. Parameters with small gradients get a proportionally larger decay than intended. Parameters with large gradients get less decay.

*   **Cosine Annealing LR scheduler is used** which automatically adjusts the learning rate during training, where the learning rate follows a cosine curve. This will help our model to converge faster.

*   **Gradient Clipping**: it is a technique to **control exploding gradients**, where we limit how large gradients are allowed to be during backpropagation. For example,

    Instead of allowing:

    ∥∇L∥=500

    We force it to be at most 1:

    ∥∇L∥≤1






In [ ]:
def train(model, trainloader, valloader=None, epochs=20, lr=0.001):
    model.to(device)
    criterion = nn.CrossEntropyLoss(label_smoothing=0.1)  # label smoothing
    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)  # LR scheduler used

    history = {'train_loss': [], 'val_loss': [], 'val_acc': []}

    for epoch in range(epochs):
        model.train()
        running_loss = 0.0

        for images, labels in trainloader:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()       # zero grad resets all gradients to zero before backpropagation.
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)  # gradient clipping
            optimizer.step()
            running_loss += loss.item() # we accumulate the losses

        scheduler.step()
        avg_train_loss = running_loss / len(trainloader)
        history['train_loss'].append(avg_train_loss)

        # Validation pass
        if valloader is not None:
            val_loss, val_acc = evaluate(model, valloader, return_metrics=True)
            history['val_loss'].append(val_loss)
            history['val_acc'].append(val_acc)
            print(f'Epoch {epoch+1:02d}/{epochs} | Train Loss: {avg_train_loss:.4f} | Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.2f}%')
        else:
            print(f'Epoch {epoch+1:02d}/{epochs} | Train Loss: {avg_train_loss:.4f}')

    return history

###  3b. Evaluation

In [ ]:
def evaluate(model, loader, return_metrics=False):
    model.eval()
    criterion = nn.CrossEntropyLoss()
    correct, total, total_loss = 0, 0, 0.0

    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            total_loss += loss.item()
            _, predicted = torch.max(outputs, 1)
            total   += labels.size(0)
            correct += (predicted == labels).sum().item()

    acc = 100.0 * correct / total
    avg_loss = total_loss / len(loader)

    if return_metrics:
        return avg_loss, acc
    print(f'Test Accuracy: {acc:.2f}%  |  Test Loss: {avg_loss:.4f}')
    return acc

###  3c. Loss Curve Plotting

In [ ]:
def plot_history(history, title='Training History'):
    epochs = range(1, len(history['train_loss']) + 1)
    fig, axes = plt.subplots(1, 2, figsize=(14, 4))

    axes[0].plot(epochs, history['train_loss'], label='Train Loss', color='steelblue')
    if history['val_loss']:
        axes[0].plot(epochs, history['val_loss'], label='Val Loss', color='tomato')
    axes[0].set_title('Loss Curve'); axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss')
    axes[0].legend(); axes[0].grid(True, alpha=0.3)

    if history['val_acc']:
        axes[1].plot(epochs, history['val_acc'], label='Val Accuracy', color='seagreen')
        axes[1].set_title('Validation Accuracy'); axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Accuracy (%)')
        axes[1].legend(); axes[1].grid(True, alpha=0.3)

    fig.suptitle(title, fontsize=14)
    plt.tight_layout()
    plt.show()

###  3d. Prediction Visualizer

In [ ]:
def imshow(img, ax, mean=CIFAR_MEAN, std=CIFAR_STD):
    """Denormalize and display a tensor image on a given axis."""
    img = img.clone().cpu().float()
    for c, (m, s) in enumerate(zip(mean, std)):
        img[c] = img[c] * s + m      # inverse normalize
    img = img.permute(1, 2, 0).numpy()
    img = np.clip(img, 0, 1)
    ax.imshow(img)
    ax.axis('off')


def show_predictions(model, loader, n=16, title='Predictions vs Ground Truth'):
    """Display n sample images with ground truth and predicted labels.
       Correct predictions shown in green, wrong in red.
    """
    model.eval()
    model.to(device)

    dataiter = iter(loader)
    images, labels = next(dataiter)
    images_dev = images.to(device)

    with torch.no_grad():
        outputs   = model(images_dev)
        _, predicted = torch.max(outputs, 1)

    predicted = predicted.cpu()
    n = min(n, len(images))
    cols = 8
    rows = (n + cols - 1) // cols

    fig, axes = plt.subplots(rows, cols, figsize=(cols * 2, rows * 2.5))
    axes = axes.flatten() if rows > 1 else axes

    for i in range(n):
        ax = axes[i] if n > 1 else axes
        imshow(images[i], ax)
        true_lbl = classes[labels[i]]
        pred_lbl = classes[predicted[i]]
        correct  = (labels[i] == predicted[i])
        color    = 'green' if correct else 'red'
        ax.set_title(f'GT: {true_lbl}\nPred: {pred_lbl}', fontsize=8,
                     color=color, fontweight='bold')

    # Hide unused axes
    for j in range(n, len(axes)):
        axes[j].set_visible(False)

    fig.suptitle(title, fontsize=13)
    plt.tight_layout()
    plt.show()

### 3e. Getting Predictions (to make confusion matrix):

In [ ]:
def get_predictions(model, dataloader, device):
    model.eval()
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for images, labels in dataloader:
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            _, preds = torch.max(outputs, 1)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    return np.array(all_labels), np.array(all_preds)

### 3f. Plotting Confusion matrix:

In [ ]:
def plot_confusion_matrix(y_true, y_pred, classes, title):
    cm = confusion_matrix(y_true, y_pred)

    plt.figure(figsize=(8,6))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=classes,
                yticklabels=classes)
    plt.xlabel("Predicted")
    plt.ylabel("Actual")
    plt.title(title)
    plt.show()

### 3g. K-Fold Cross Validation

Uses Stratified K-Fold (via `sklearn`) on the **training set** to get a robust estimate of model performance across different data splits. Final evaluation is always done on the held-out test set.

> **Why K-Fold?** With K=5, each fold uses 80% for training and 20% for validation, cycling through all data. The mean ± std of fold accuracies indicates how stable the model is.

In [ ]:
def run_kfold(model_fn, dataset, k=5, epochs=10, batch_size=128):
    """
    Run K-Fold cross validation.
    model_fn : callable with no args that returns a fresh model instance.
    dataset  : full training dataset.
    """
    kf = KFold(n_splits=k, shuffle=True, random_state=42)
    fold_accs  = []
    fold_losses = []
    indices = list(range(len(dataset)))

    for fold, (train_idx, val_idx) in enumerate(kf.split(indices), 1):
        print(f'\n{'='*50}')
        print(f'Fold {fold}/{k}  |  Train: {len(train_idx)}  Val: {len(val_idx)}')
        print('='*50)

        train_subset = Subset(dataset, train_idx)
        val_subset   = Subset(dataset, val_idx)

        fold_train_loader = DataLoader(train_subset, batch_size=batch_size, shuffle=True,  num_workers=2, pin_memory=True)
        fold_val_loader   = DataLoader(val_subset,   batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True)

        model = model_fn().to(device)
        history = train(model, fold_train_loader, fold_val_loader, epochs=epochs)

        val_loss, val_acc = evaluate(model, fold_val_loader, return_metrics=True)
        fold_accs.append(val_acc)
        fold_losses.append(val_loss)
        print(f'Fold {fold} Final  |  Val Acc: {val_acc:.2f}%  |  Val Loss: {val_loss:.4f}')

    print(f'\n{"="*50}')
    print(f'K-Fold Results ({k} folds, {epochs} epochs/fold)')
    print(f'  Accuracies  : {[f"{a:.2f}" for a in fold_accs]}')
    print(f'  Mean Acc    : {np.mean(fold_accs):.2f}% ± {np.std(fold_accs):.2f}%')
    print(f'  Mean Loss   : {np.mean(fold_losses):.4f} ± {np.std(fold_losses):.4f}')
    print('='*50)

    # Plot per-fold accuracy
    plt.figure(figsize=(8, 4))
    plt.bar(range(1, k+1), fold_accs, color='steelblue', edgecolor='white')
    plt.axhline(np.mean(fold_accs), color='tomato', linestyle='--', label=f'Mean: {np.mean(fold_accs):.2f}%')
    plt.xlabel('Fold'); plt.ylabel('Validation Accuracy (%)')
    plt.title(f'{k}-Fold Cross Validation Accuracy'); plt.legend(); plt.tight_layout()
    plt.show()

    return fold_accs, fold_losses

###  4a. ANN (Baseline model)

*   **Layer 1: Flatten Layer -** We flatten the 3D (32 by 32 by 3) image as traditional neural networks do not understand spatial information. The model just sees: **3072 unrelated numbers**
*   **Layer 2: Linear Layer -** Brings 3072 pixels down to 1024 pixels.
*   **Layer 3: ReLU Activation Layer**
*   **Layer 4: Batch Normalisation Layer**
*   **Layer 5: Dropout Layer -** A dropout layer which drops 50% of the neurons.
*   **Layer 6: Linear Layer -** Brings 1024 pixels down to 512 pixels.
*   **Layer 7: ReLU Activation Layer**
*   **Layer 8: Batch Normalisation Layer**
*   **Layer 9: Dropout Layer**

In [ ]:
class ANN(nn.Module):
    def __init__(self):
        super().__init__()
        self.model = nn.Sequential(
            nn.Flatten(),
            nn.Linear(32*32*3, 1024),
            nn.ReLU(),
            nn.BatchNorm1d(1024),
            nn.Dropout(0.5),
            nn.Linear(1024, 512),
            nn.ReLU(), nn.BatchNorm1d(512),
            nn.Dropout(0.5),
            nn.Linear(512, 10)
        )
    def forward(self, x): return self.model(x)

ann_model = ANN()
ann_history = train(ann_model, trainloader, testloader, epochs=50)
plot_history(ann_history, 'ANN Training History')
evaluate(ann_model, testloader)
show_predictions(ann_model, testloader, title='ANN : Predictions vs Ground Truth')

###  4b. ANN (Improved model)

This is an improvement from the baseline model above: where we are trying to make the network deeper + wider.

This will make the model learn more complex features and thus improve prediction.

Here we are transitioning from:

3072 -> 2048 -> 1024 -> 512 -> 256 -> 10 (pixels)

compared to earlier architecture which transitioned from:

3072 -> 1024 -> 512 -> 10 (pixels)

What are the improvements?

* **Making NN deeper:** Adding more layers helps the model learn better as each layer understands a much detailed feature and helps make the predictions better.
* **Making NN wider:** Adding more neurons per layer makes the NN wider. Here it is represented by **out_features** in the layers:

    1024 neurons → 1024 learned features

    2048 neurons → 2048 learned features

    So, instead of compressing pixels early, we allow richer feature expansion first, preserving information better.

    So increasing the out_features increases the learning capacity thus improving the predictions.

* **Dropout Change:** Earlier, we were using **dropout value: 0.5** twice. *50% can be heavy dropping* and can lead to missing out on some important features.

    Hence, herewe tried reducing dropout while moving from one layer to another:

    0.4 -> 0.4 -> 0.3


In [ ]:
class ImprovedANN(nn.Module):
    def __init__(self):
        super().__init__()
        self.model = nn.Sequential(
            nn.Flatten(),

            nn.Linear(32*32*3, 2048),
            nn.ReLU(),
            nn.BatchNorm1d(2048),
            nn.Dropout(0.4),

            nn.Linear(2048, 1024),
            nn.ReLU(),
            nn.BatchNorm1d(1024),
            nn.Dropout(0.4),

            nn.Linear(1024, 512),
            nn.ReLU(),
            nn.BatchNorm1d(512),
            nn.Dropout(0.3),

            nn.Linear(512, 256),
            nn.ReLU(),

            nn.Linear(256, 10)
        )

    def forward(self, x):
        return self.model(x)

ann_improv_model = ImprovedANN()
ann_new_history = train(ann_improv_model, trainloader, testloader, epochs=50)
plot_history(ann_new_history, 'ANN Training History')
evaluate(ann_improv_model, testloader)
show_predictions(ann_improv_model, testloader, title='ANN : Predictions vs Ground Truth')

### Confusion Matrix to compare **ANN Baseline model vs the Improved ANN model**

In [ ]:
# ANN Baseline model
y_true_ann_base, y_pred_ann_base = get_predictions(ann_model, testloader, device)
plot_confusion_matrix(y_true_ann_base, y_pred_ann_base, classes,
                      "ANN Baseline Confusion Matrix")


# Improved ANN model
y_true_ann_imp, y_pred_ann_imp = get_predictions(ann_improv_model, testloader, device)
plot_confusion_matrix(y_true_ann_imp, y_pred_ann_imp, classes,
                      "Improved ANN Confusion Matrix")

### K-Fold Cross Validation for Improved ANN Model

In [ ]:
# 5-Fold Cross Validation using the Improved ANN model architecture
fold_accs, fold_losses = run_kfold(
    model_fn=ImprovedANN,
    dataset=full_trainset,
    k=5,
    epochs=10,
    batch_size=128
)

###  5. CNN (Baseline)

As mentioned: input image is of dimensions: *32 by 32 by 3*

The architecture of the baseline model is:
* **1st convolutional block:**
  * **Layer 1: Convolution Layer (32 channels):** Applies 32 filters of size 3×3 with padding=1

    Output Dimensions: 32 × 32 × 32

  * **Layer 2: Batch Normalisation (32 channels):**

    Output Dimensions: 32 × 32 × 32

  * **Layer 3: ReLU Activation layer**

  * **Layer 4: Max Pooling (2×2):**

    Output Dimensions: 32 × 16 × 16

  * **Layer 5: Dropout2D (value=0.25):**

* **2nd convolutional block:**
  * **Layer 6: Convolution (64 channels):**

    Output Dimensions: 64 × 16 × 16

  * **Layer 7: Batch Normalisation (64 channels)**

  * **Layer 8: ReLU**

  * **Layer 9: Max Pooling (2×2):**
     
    Output Dimensions: 64 × 8 × 8

  * **Layer 10: Dropout2D (25%)**

* **3rd convolutional block:**
  * **Layer 11: Convolution (128 channels):**

    Output Dimensions: 128 × 8 × 8

  * **Layer 12: Batch Normalisation (128 channels)**

  * **Layer 13: ReLU**

  * **Layer 14: Max Pooling (2×2):**

    Output Dimensions: 128 × 4 × 4

  * **Layer 15: Dropout2D (25%)**

* **Fully Connected Layer**

  * **Layer 16: Flatten Layer:** This converts:

    128 × 4 × 4 into: 2048 features

  * **Layer 17: Linear Layer (2048 → 256):** This compresses extracted features into 256 high-level representations.

  * **Layer 18: ReLU Activation Layer**

  * **Layer 19: Batch Normalisation (256)**

  * **Layer 20: Dropout (value=0.5)**

  * **Layer 21: Final Linear Layer (256 → 10):** This layer **outputs 10 class scores** corresponding to CIFAR-10 classes.

In [ ]:
class CNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(3,   32, 3, padding=1), nn.BatchNorm2d(32),  nn.ReLU(), nn.MaxPool2d(2), nn.Dropout2d(0.25),
            nn.Conv2d(32,  64, 3, padding=1), nn.BatchNorm2d(64),  nn.ReLU(), nn.MaxPool2d(2), nn.Dropout2d(0.25),
            nn.Conv2d(64, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(), nn.MaxPool2d(2), nn.Dropout2d(0.25),
        )
        self.fc = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128*4*4, 256), nn.ReLU(), nn.BatchNorm1d(256), nn.Dropout(0.5),
            nn.Linear(256, 10)
        )
    def forward(self, x): return self.fc(self.conv(x))

cnn_model = CNN()
cnn_history = train(cnn_model, trainloader, testloader, epochs=50)
plot_history(cnn_history, 'CNN Training History')
evaluate(cnn_model, testloader)
show_predictions(cnn_model, testloader, title='CNN : Predictions vs Ground Truth')

## 6. Architecture Improvements

Each cell below adds one improvement on top of the previous:

| Step | Improvement | Expected Gain |
|------|-------------|---------------|
| Step 1 | Deeper CNN + Global Average Pooling | Fewer params, thus better generalization |
| Step 2 | Residual (Skip) Connections | This leads to easier gradient flow |
| Step 3 | Depthwise Separable Convolutions | Efficiency + regularization |
| Step 4 | Mixup data augmentation | Stronger regularization |


###  Step 1 : Deeper CNN + Global Average Pooling (GAP)
As we did with ANN, we will try to improve predictions by adding layers, thus making the CNN deeper.

### # Why Global Average Pooling and how does it improve our model?

* **In the original CNN architecture discussed previously:**

  *  After the flatten layer: 128 × 4 × 4 → 2048.
  *  So now the fully connected layer sees: **2048 input neurons**
  *  And later comes the Big Fully Connected Layer which is **Linear(2048 → 256)**
  *  And hence the number of parameters: **2048×256=5,24,288 (plus 256 biases)**
  *  Plus, we have another **fully connected linear layer Linear(256 → 10)**

* **But with GAP:**

  * We start from the same output: 128 × 4 × 4. But instead of flattening,
  * We have the averaging layer: where we average each channel

    * For each of the 128 channels:
    * We take average of all 16 values (4×4).
    * Now we only have 128 numbers.

  * Final Linear Layer: **Linear(128 → 10)**

    So here number of parameters: 128×10= **1,280 parameters**

  **This is a huge reduction in the number of parameters, and thus improves accuracy.**

### **Apart from this:**
**Without GAP:**

The network memorizes exact positions as the image is flattened and each pixel is treated as an individual element.

**With GAP:**

The network measures how strongly each feature exists in the image, that is, it helps model learn presence of features in the entire image.

In [ ]:
class DeepCNN_GAP(nn.Module):

    def __init__(self, num_classes=10):
        super().__init__()
        self.features = nn.Sequential(
            # Block 1
            nn.Conv2d(3,  64, 3, padding=1), nn.BatchNorm2d(64),  nn.ReLU(),
            nn.Conv2d(64, 64, 3, padding=1), nn.BatchNorm2d(64),  nn.ReLU(),
            nn.MaxPool2d(2), nn.Dropout2d(0.2),
            # Block 2
            nn.Conv2d(64, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(),
            nn.Conv2d(128,128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(),
            nn.MaxPool2d(2), nn.Dropout2d(0.2),
            # Block 3
            nn.Conv2d(128, 256, 3, padding=1), nn.BatchNorm2d(256), nn.ReLU(),
            nn.Conv2d(256, 256, 3, padding=1), nn.BatchNorm2d(256), nn.ReLU(),
        )
        self.gap = nn.AdaptiveAvgPool2d(1)   # Global Average Pooling => (B, 256, 1, 1)
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(0.5),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        return self.classifier(self.gap(self.features(x)))

model_step1 = DeepCNN_GAP()


In [ ]:
hist1 = train(model_step1, trainloader, testloader, epochs=50)
plot_history(hist1, 'Step 1: DeepCNN + GAP')
evaluate(model_step1, testloader)
show_predictions(model_step1, testloader, title='Step 1 : DeepCNN+GAP Predictions')


###  Step 2 : ResCNN : Inspired from ResNet:
We added residual (or skip) connections between layers. These skip paths preserve information across layers that might have been lost as the neural network gets deeper. Also, when some layers are skipped via these connections, it prevents vanishing gradients.

ResCNN (our model here) also includes global average pooling to include the improvements mentioned previously.

In [ ]:
class ResBlock(nn.Module):

    def __init__(self, in_ch, out_ch, stride=1):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, stride=stride, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
        )
        self.skip = nn.Sequential()
        if stride != 1 or in_ch != out_ch:
            self.skip = nn.Sequential(
                nn.Conv2d(in_ch, out_ch, 1, stride=stride, bias=False),
                nn.BatchNorm2d(out_ch)
            )
        self.relu = nn.ReLU(inplace=True)

    def forward(self, x):
        return self.relu(self.conv(x) + self.skip(x))


class ResCNN(nn.Module):

    def __init__(self, num_classes=10):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv2d(3, 64, 3, padding=1, bias=False),
            nn.BatchNorm2d(64), nn.ReLU(inplace=True)
        )
        self.layer1 = ResBlock(64,  128, stride=2)
        self.layer2 = ResBlock(128, 256, stride=2)
        self.layer3 = ResBlock(256, 512, stride=2)
        self.gap = nn.AdaptiveAvgPool2d(1)
        self.fc  = nn.Sequential(nn.Flatten(), nn.Dropout(0.4), nn.Linear(512, num_classes))

    def forward(self, x):
        x = self.stem(x)
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        return self.fc(self.gap(x))

model_step2 = ResCNN()
hist2 = train(model_step2, trainloader, testloader, epochs=50)
plot_history(hist2, 'Step 2: ResCNN (Residual Connections)')
evaluate(model_step2, testloader)
show_predictions(model_step2, testloader, title='Step 2 : ResCNN Predictions')

###  Step 3 : Depthwise Separable Convolutions
Another improvement we tried is
**replacing standard 3×3 convolutions** with **depthwise-separable ones** for fewer parameters (Inspired from **VGG-16 architecture**)

**Parameter reduction:**

* **With 3×3 convolutions:** we have 128 × 256 × 3×3 = **294,912 parameters** per layer.

* **With Depthwise convolutions:** one 3×3 filter per input channel is applied independently.

    So 128 channels produce 128 filtered maps.

    which means 128 × 3×3 = **1,152 parameters**
    
* This is followed by **Pointwise convolutions:** a 1×1 convolution is applied that mixes the channels to produce the 256 outputs.

     which means 28 × 256 × 1×1 = **32,768 parameters** which is still way lesser than general 3×3 convolution.

In [ ]:
class DepthwiseSep(nn.Module):
    """Depthwise Separable Conv block: depthwise -> pointwise."""
    def __init__(self, in_ch, out_ch, stride=1):
        super().__init__()
        self.block = nn.Sequential(
            # Depthwise
            nn.Conv2d(in_ch, in_ch, 3, stride=stride, padding=1, groups=in_ch, bias=False),
            nn.BatchNorm2d(in_ch), nn.ReLU(inplace=True),
            # Pointwise
            nn.Conv2d(in_ch, out_ch, 1, bias=False),
            nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),
        )
    def forward(self, x): return self.block(x)


class EfficientCNN(nn.Module):
    """Lightweight CNN using depthwise separable convolutions."""
    def __init__(self, num_classes=10):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1, bias=False), nn.BatchNorm2d(32), nn.ReLU(inplace=True),
            DepthwiseSep(32,  64, stride=2),
            DepthwiseSep(64, 128, stride=2),
            DepthwiseSep(128, 256, stride=2),
            DepthwiseSep(256, 512, stride=1),
            nn.AdaptiveAvgPool2d(1),
        )
        self.fc = nn.Sequential(nn.Flatten(), nn.Dropout(0.4), nn.Linear(512, num_classes))

    def forward(self, x): return self.fc(self.net(x))

model_step3 = EfficientCNN()
hist3 = train(model_step3, trainloader, testloader, epochs=50)
plot_history(hist3, 'Step 3: EfficientCNN (Depthwise Separable Convolutions)')
evaluate(model_step3, testloader)
show_predictions(model_step3, testloader, title='Step 3 : EfficientCNN Predictions')

###  Step 4 : Mixup Data Augmentation
Mixup Data Augmentation is one of the most powerful augmentation methods.

So instead of training on:

*(image A, label A)*

Mixup trains on:

*(mixed image, mixed label)*

This smoothening of labels essentially discourages the model from becoming overly confident in any single prediction.

Mixup:

* Creates smooth transitions between classes
* Prevents memorization

It forces the model to behave linearly between samples. Also, this way, the model becomes more robust, i.e, it's better at handling ambiguous or borderline cases, and it generalizes better to unseen data.

Here, we will combine ResCNN with Mixup data augmentation to see if it improves model accuracy or not.

In [ ]:
def mixup_data(x, y, alpha=0.4):

    if alpha > 0:
        lam = np.random.beta(alpha, alpha)
    else:
        lam = 1.0
    batch_size = x.size(0)
    idx = torch.randperm(batch_size).to(x.device)
    mixed_x = lam * x + (1 - lam) * x[idx]
    y_a, y_b = y, y[idx]
    return mixed_x, y_a, y_b, lam


def mixup_criterion(criterion, pred, y_a, y_b, lam):
    return lam * criterion(pred, y_a) + (1 - lam) * criterion(pred, y_b)


def train_mixup(model, trainloader, valloader=None, epochs=20, lr=0.001, alpha=0.4):
    model.to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    history   = {'train_loss': [], 'val_loss': [], 'val_acc': []}

    for epoch in range(epochs):
        model.train()
        running_loss = 0.0
        for images, labels in trainloader:
            images, labels = images.to(device), labels.to(device)
            images, y_a, y_b, lam = mixup_data(images, labels, alpha=alpha)
            optimizer.zero_grad()
            outputs = model(images)
            loss = mixup_criterion(criterion, outputs, y_a, y_b, lam)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            running_loss += loss.item()
        scheduler.step()
        avg_loss = running_loss / len(trainloader)
        history['train_loss'].append(avg_loss)

        if valloader is not None:
            val_loss, val_acc = evaluate(model, valloader, return_metrics=True)
            history['val_loss'].append(val_loss)
            history['val_acc'].append(val_acc)
            print(f'Epoch {epoch+1:02d}/{epochs} | Train Loss: {avg_loss:.4f} | Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.2f}%')
        else:
            print(f'Epoch {epoch+1:02d}/{epochs} | Train Loss: {avg_loss:.4f}')
    return history

model_step4 = ResCNN()
hist4 = train_mixup(model_step4, trainloader, testloader, epochs=50, alpha=0.4)
plot_history(hist4, 'Step 4: ResCNN + MixUp Augmentation')
evaluate(model_step4, testloader)

In [ ]:

show_predictions(model_step4, testloader, title='Step 4 — ResCNN+Mixup Predictions')

### Confusion Matrix to compare **CNN Baseline model vs the ResCNN model**

In [ ]:
# CNN Baseline
y_true_cnn_base, y_pred_cnn_base = get_predictions(cnn_model, testloader, device)
plot_confusion_matrix(y_true_cnn_base, y_pred_cnn_base, classes,
                      "CNN Baseline Confusion Matrix")


# Best CNN
y_true_best, y_pred_best = get_predictions(model_step2, testloader, device)
plot_confusion_matrix(y_true_best, y_pred_best, classes,
                      "Best CNN Confusion Matrix")

## K-Fold Cross Validation for ResCNN

In [ ]:
# 5-Fold Cross Validation using the ResCNN architecture
fold_accs, fold_losses = run_kfold(
    model_fn=ResCNN,
    dataset=full_trainset,
    k=5,
    epochs=10,
    batch_size=128
)

## 6. Final Model Comparison

In [ ]:
# Collect final test accuracies for all models
results = {}
for name, model in [
    ('ANN (Baseline)',         ann_model),
    ('ANN (Improved Model)', ann_improv_model),
    ('CNN (Baseline)',         cnn_model),
    ('DeepCNN + GAP',          model_step1),
    ('ResCNN',                 model_step2),
    ('EfficientCNN (DWS)',     model_step3),
    ('ResCNN + Mixup',         model_step4),
]:
    _, acc = evaluate(model, testloader, return_metrics=True)
    results[name] = acc
    print(f'{name:30s}  Acc: {acc:.2f}%')

# Bar chart
plt.figure(figsize=(10, 5))
bars = plt.barh(list(results.keys()), list(results.values()),
                color=['#e07b7b','#e0a77b','#b0d06b','#6bc1d0','#6b7cd0','#a06bd0'])
for bar, val in zip(bars, results.values()):
    plt.text(bar.get_width() + 0.3, bar.get_y() + bar.get_height()/2,
             f'{val:.2f}%', va='center', fontsize=9)
plt.xlabel('Test Accuracy (%)'); plt.title('Model Comparison — CIFAR-10 Test Accuracy')
plt.xlim(0, 105); plt.tight_layout(); plt.show()